In [65]:
import numpy as np
import pickle
import casadi as ca
import time

num_var = 20
num_ineq = 10
num_eq = 10
num_examples = 2
seed = 2025

print("Nonsmooth nonconvex SOCP problem with {} variables, {} inequalities, {} equalities and {} examples".format(num_var, num_ineq, num_eq, num_examples))
np.random.seed(seed)
Q = np.diag(np.random.rand(num_var)*0.5)
p = np.random.uniform(-1, 1, num_var)
A = np.random.uniform(-1, 1, size=(num_eq, num_var))
X = np.random.uniform(-1, 1, size=(num_examples, num_eq))
XL = X.min(axis=0)
XU = X.max(axis=0)

L = np.ones((num_var))*-5
U = np.ones((num_var))*5
x0 = np.random.uniform(-1, 1, size=(num_var))
G = []
h = []
C = []
d = []
for i in range(num_ineq):
    G.append(np.random.uniform(-1, 1, size=(num_ineq, num_var)))
    h.append(np.random.uniform(-1, 1, size=(num_ineq)))
    C.append(np.random.uniform(-1, 1, size=(num_var)))
    d.append(np.linalg.norm(G[i] @ x0 + h[i], 2) - C[i].T @ x0)
data = {'Q':Q,
        'p':p,
        'A':A,
        'X':X,
        'G':np.array(G),
        'h':np.array(h),
        'C':np.array(C),
        'd':np.array(d),
        'YL':L,
        'YU':U,
        'XL':XL,
        'XU':XU,
        'Y':[]}
Y = []
for n in range(num_examples):
    Xi = X[n]
    y = ca.MX.sym('y_var', num_var)
    t = ca.MX.sym('t_var')

    obj_func = 0.5 * ca.mtimes(y.T, ca.mtimes(Q, y)) + ca.dot(p, ca.sin(y)) + 0.1*t

    eq_constraints = A @ y - Xi
    soc = ca.dot(y, y) - t**2
    ineq_constraints = []
    for i in range(num_ineq):
        ineq_constraints.append(ca.norm_2(G[i] @ ca.cos(y) + h[i]) - (ca.dot(C[i], y) + d[i]))
    ineq_constraints.append(soc)
    ineq_constraints = ca.vertcat(*ineq_constraints)
    
    nlp = {'x': ca.vertcat(y, t), 'f': obj_func, 'g': ca.vertcat(eq_constraints, ineq_constraints)}
    opts = {'ipopt.print_level': 0, 'print_time': 0, }
    solver = ca.nlpsol('solver', 'ipopt', nlp, opts)
    # Define bounds for variables and constraints
    lbg = np.concatenate([np.zeros(num_eq), -np.inf * np.ones(num_ineq+1)])
    ubg = np.concatenate([np.zeros(num_eq), np.zeros(num_ineq+1)])
    lbx = np.concatenate([L, [0]])
    ubx = np.concatenate([U, [np.inf]])
    # Solve the NLP
    start_time = time.time()
    res = solver(lbg=lbg, ubg=ubg, lbx=lbx, ubx=ubx)
    print("Time for example {}: {:.4f} seconds".format(n, time.time() - start_time))
    # check if the solver converged
    if solver.stats()['success']:
        sol_x = res['x'].full().flatten()
        Y.append(sol_x[:-1])
    else:
        print("Solver failed to converge")
        break

    print("Example {}: Objective value: {}".format(n, res['f'].full().flatten()[0]))

data['Y'] = np.array(Y)


i = 0
det_min = 0
best_partial = 0
while i < 1000:
    np.random.seed(i)
    partial_vars = np.random.choice(num_var, num_var - num_eq, replace=False)
    other_vars = np.setdiff1d(np.arange(num_var), partial_vars)
    _, det = np.linalg.slogdet(A[:, other_vars])
    if det>det_min:
        det_min = det
        best_partial = partial_vars
    i += 1
print('best_det', det_min)
data['best_partial'] = best_partial


# with open("datasets/nonsmooth_nonconvex/socp/random{}_socp_dataset_var{}_ineq{}_eq{}_ex{}".format(seed, num_var, num_ineq, num_eq, num_examples), 'wb') as f:
#     pickle.dump(data, f)

Nonsmooth nonconvex SOCP problem with 20 variables, 10 inequalities, 10 equalities and 2 examples
Time for example 0: 0.0134 seconds
Example 0: Objective value: 2.9323267488167493
Time for example 1: 0.0227 seconds
Example 1: Objective value: 0.24978387468803653
best_det 3.171747174916341


In [38]:
# print all input and output data
for key, value in data.items():
    print(key, value)

Q [[0.06774408 0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.44392585 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.         0.46630282 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.         0.         0.22278408 0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.        ]
 [0.         0.         0.         0.         0.19411777 0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0

In [79]:
import numpy as np
from scipy.optimize import minimize, NonlinearConstraint, Bounds


# -----------------------------
# Objective
# -----------------------------
def objective(z):
    y = z[:-1]
    t = z[-1]
    return 0.5 * y @ Q @ y + p @ np.sin(y) + 0.1 * t

# -----------------------------
# Constraints
# -----------------------------
def equality_constraint(z, x):
    y = z[:-1]
    return A @ y - x

def soc_constraint(z):
    y = z[:-1]
    t = z[-1]
    return np.dot(y, y) - t**2

def nonsmooth_constraints(z):
    y = z[:-1]
    return np.array([
        np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        for i in range(num_ineq)
    ])


# -----------------------------
# Solve for each X
# -----------------------------
Y2 = []

for n in range(num_examples):
    x = X[n]

    eq_con = NonlinearConstraint(
        lambda z, x=x: equality_constraint(z, x),
        lb=np.zeros(num_eq),
        ub=np.zeros(num_eq)
    )

    soc_con = NonlinearConstraint(
        soc_constraint,
        lb=-np.inf,
        ub=0.0
    )

    nonsmooth_con = NonlinearConstraint(
        nonsmooth_constraints,
        lb=-np.inf * np.ones(num_ineq),
        ub=np.zeros(num_ineq)
    )

    bounds = Bounds(
        np.concatenate([L, [0.0]]),
        np.concatenate([U, [np.inf]])
    )

    z0 = np.concatenate([np.zeros(num_var), [1.0]])

    res = minimize(
        objective,
        z0,
        method="SLSQP",
        bounds=bounds,
        constraints=[eq_con, soc_con, nonsmooth_con],
        options={"ftol": 1e-9, "maxiter": 500}
    )

    if not res.success:
        print(f"Failed at sample {n}: {res.message}")
        break

    Y2.append(res.x[:-1])

    print(f"Sample {n}, obj = {res.fun:.6f}")

Y2 = np.array(Y2)
print("Finished SciPy optimization")


Sample 0, obj = -0.336906
Sample 1, obj = 0.249784
Finished SciPy optimization


In [44]:
x0

array([-0.34439209,  0.84286058,  0.48770127,  0.49717519,  0.73441153,
        0.43446241, -0.37946738,  0.54560269,  0.59752861, -0.56180473,
       -0.03144527,  0.47898628,  0.40148437, -0.33991818, -0.73305877,
       -0.58050767,  0.8063994 , -0.94154894, -0.01380367, -0.24091042])

In [45]:
z0

array([-0.34439209,  0.84286058,  0.48770127,  0.49717519,  0.73441153,
        0.43446241, -0.37946738,  0.54560269,  0.59752861, -0.56180473,
       -0.03144527,  0.47898628,  0.40148437, -0.33991818, -0.73305877,
       -0.58050767,  0.8063994 , -0.94154894, -0.01380367, -0.24091042,
        1.        ])

In [ ]:
# print solutions
print("Y:", Y)
print("Y2:", Y2)
print("Difference between Y and Y2:", np.linalg.norm(Y - Y2))

Y: [array([ 1.63863986,  1.28348088,  1.85176301,  0.58227158,  0.7558684 ,
       -1.15472894, -0.98004736, -0.89615888, -0.85284814,  1.14047555,
        1.5550492 ,  0.43478538,  0.75565446, -1.64136649,  1.72442895,
       -1.63854521, -1.18241496, -1.25518649, -0.29201938,  1.28108311]), array([ 0.87862384, -0.83012591,  1.44950098, -0.38640913, -1.05420158,
        0.66492192,  0.84330134, -1.5499003 ,  1.29407301,  0.93680569,
        1.64809961, -1.92350085, -1.73095207, -1.40480768,  2.10647284,
       -1.91655208,  1.58595394, -1.51509141, -1.00260693,  3.63758259])]
Y2: [[ 0.49885727 -0.51789351  1.30100492  0.2020102  -1.17174325  1.76296037
  -0.54196973 -1.49628448  1.50244175  0.99236714  1.22843285 -1.45766139
  -1.78227607 -1.62685568  2.0750654  -1.72108811  1.48886077 -0.91444459
  -1.05514151  3.01476196]
 [ 2.00409177  1.39532551  1.16263652 -0.42614009  0.45306037 -1.50935486
  -0.59308761 -0.6052541  -1.35252861  1.18611353  1.76876812  0.93120992
   0.98092968 -

In [ ]:
def constraint_violation(y, t, x):
    # Equality violation
    eq_violation = np.linalg.norm(A @ y - x, ord=2)

    # Inequality violations
    ineq_vals = []
    for i in range(num_ineq):
        gi = np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        ineq_vals.append(max(0.0, gi))

    soc = np.dot(y, y) - t**2
    ineq_vals.append(max(0.0, soc))

    ineq_vals = np.array(ineq_vals)

    # Bound violations
    lb_violation = np.maximum(0.0, L - y)
    ub_violation = np.maximum(0.0, y - U)
    t_violation  = max(0.0, -t)

    bound_violation = np.linalg.norm(
        np.concatenate([lb_violation, ub_violation, [t_violation]]),
        ord=2
    )

    return {
        "eq_l2": eq_violation,
        "ineq_max": ineq_vals.max(),
        "ineq_l2": np.linalg.norm(ineq_vals, ord=2),
        "bound_l2": bound_violation
    }
def grad_objective(y, t):
    grad_y = Q @ y + p * np.cos(y)
    grad_t = np.array([0.1])
    return np.concatenate([grad_y, grad_t])
def jacobian_eq():
    J = np.zeros((num_eq, num_var + 1))
    J[:, :num_var] = A
    return J

def jacobian_soc(y, t):
    J = np.zeros(num_var + 1)
    J[:num_var] = 2 * y
    J[-1] = -2 * t
    return J

def jacobian_nonsmooth(y, i):
    v = G[i] @ np.cos(y) + h[i]
    norm_v = np.linalg.norm(v)

    if norm_v < 1e-8:
        return np.zeros(num_var + 1)

    J = np.zeros(num_var + 1)
    J[:num_var] = (
        -(G[i].T @ (v / norm_v)) * np.sin(y) - C[i]
    )
    return J
def kkt_residual(y, t, x):
    grad_f = grad_objective(y, t)

    # Active constraints
    J = []
    rhs = -grad_f

    # Equality constraints
    J.append(jacobian_eq())

    # Active inequalities
    for i in range(num_ineq):
        gi = np.linalg.norm(G[i] @ np.cos(y) + h[i]) - (C[i] @ y + d[i])
        if gi > -1e-6:
            J.append(jacobian_nonsmooth(y, i)[None, :])

    soc = np.dot(y, y) - t**2
    if soc > -1e-6:
        J.append(jacobian_soc(y, t)[None, :])

    if not J:
        return np.linalg.norm(grad_f)

    J = np.vstack(J)

    # Least-squares multipliers
    try:
        lam, *_ = np.linalg.lstsq(J.T, rhs, rcond=None)
        res = grad_f + J.T @ lam
        return np.linalg.norm(res)
    except np.linalg.LinAlgError:
        return np.inf


In [ ]:
y = Y[n]      # IPOPT solution
t = t_ipopt   # from solver
x = X[n]

viol = constraint_violation(y, t, x)
kkt  = kkt_residual(y, t, x)

print("Constraint violation:", viol)
print("KKT residual:", kkt)


NameError: name 't_ipopt' is not defined